In [1]:
import scipy.io
import io

import numpy as np
from matplotlib import pyplot as plt
import numpy.random as rng
import pandas as pd

import os
import math

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def preprocess_rewards(left_reward_ts_, right_reward_ts_, begin_run_t, end_run_t, time_run):
    def process_timestamps(raw_ts):
        """
        Process and filter timestamps, keeping the indices of the rewards within the time_loc range.
        """
        processed_ts_with_indices = []
        # Ensure all timestamps are floats
        if isinstance(raw_ts[0], str):
            raw_ts = [float(t[:14]) for t in raw_ts]
        
        # Keep timestamps within the time range
        for reward_ts in raw_ts:
            if begin_run_t <= reward_ts <= end_run_t:
                # Find the index of the time_loc closest to the reward timestamp
                closest_idx = np.abs(time_run - reward_ts).argmin()
                processed_ts_with_indices.append((reward_ts, closest_idx))  # Tuple of (reward timestamp, index in time_loc)
        
        return processed_ts_with_indices

    # Process left and right reward timestamps
    left_reward_ts = process_timestamps(left_reward_ts_)
    right_reward_ts = process_timestamps(right_reward_ts_)

    return left_reward_ts, right_reward_ts

In [3]:
def plot_run_with_puffs(x_run, time_run, puff_x_location, puff_ts):
    """
    Plots the x-location vs. time during the run and marks the airpuff events.

    Parameters:
    - x_run (array-like): The x-locations of the run.
    - time_run (array-like): The time points corresponding to the x-locations.
    - puff_x_location (array-like): The x-locations of airpuff events.
    - puff_ts (array-like): The time points of the airpuff events.
    """
    plt.figure(figsize=(4, 8))
    plt.plot(x_run, time_run, label="X-Location", color="blue", alpha=0.7)
    plt.scatter(puff_x_location, puff_ts, color='red', s=50, label="Airpuff Events")
    plt.title('Time puffs during the run')
    plt.xlabel('Location x')
    plt.ylabel('Time (ms)')
    plt.legend()
    plt.show()


In [4]:
def get_reward_x_values(x_run, reward_ts_with_indices):
    """
    Retrieve the x values corresponding to reward timestamps using their indices.

    Parameters:
    - x_run: A Pandas Series or NumPy array of x values (positions) for the run.
    - reward_ts_with_indices: List of tuples [(timestamp, index)] from preprocess_rewards.

    Returns:
    - reward_x_values: List of (timestamp, x_value) tuples.
    """
    reward_x_values = []

    for timestamp, idx in reward_ts_with_indices:
        # Ensure index is valid
        if 0 <= idx < len(x_run):
            x_value = x_run.iloc[idx]  # Use .iloc for positional indexing
            reward_x_values.append((timestamp, x_value))
        else:
            reward_x_values.append((timestamp, None))  # Handle invalid indices gracefully

    return reward_x_values

In [5]:
def plot_rewards_on_run(x_run, time_run, left_rewards, right_rewards):
    """
    Plot the X-location over time and overlay left and right reward time stamps.

    Parameters:
    - x_run: X-location during the run.
    - time_run: Corresponding times for the X-location.
    - left_rewards: List of tuples with reward timestamps and their indices for left reward.
    - right_rewards: List of tuples with reward timestamps and their indices for right reward.
    """
    # Get the reward X values using the provided function
    left_reward_x_values = get_reward_x_values(x_run, left_rewards)
    right_reward_x_values = get_reward_x_values(x_run, right_rewards)

    # Extract the timestamps and x-values for the left and right rewards
    left_reward_timestamps = [reward[0] for reward in left_reward_x_values]
    left_reward_locations = [reward[1] for reward in left_reward_x_values]

    right_reward_timestamps = [reward[0] for reward in right_reward_x_values]
    right_reward_locations = [reward[1] for reward in right_reward_x_values]

    # Create the plot
    plt.figure(figsize=(10, 6))

    # Plot the X-location vs. Time
    plt.plot(time_run, x_run, label="X-Location", color="blue", alpha=0.7)

    # Plot the left and right reward time stamps
    plt.scatter(left_reward_timestamps, left_reward_locations, color='green', s=50, label="Left Rewards", marker='o')
    plt.scatter(right_reward_timestamps, right_reward_locations, color='orange', s=50, label="Right Rewards", marker='x')

    plt.title('X-Location vs Time with Left and Right Reward Time Stamps')
    plt.xlabel('Time (ms)')
    plt.ylabel('X-Location')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [6]:
def exclude_airpuff_periods(left_rewards, right_rewards, airpuff_ts):
    """
    Exclude the periods where air puffs occurred between reward timestamps.
    
    Parameters:
    - left_rewards: List of left reward timestamps and indices.
    - right_rewards: List of right reward timestamps and indices.
    - airpuff_ts: List of air puff timestamps.
    
    Returns:
    - safe_runs: A list of tuples containing start and end timestamps and indices for each safe run.
    """
    # Combine left and right rewards and sort by timestamp
    all_rewards = sorted(left_rewards + right_rewards, key=lambda x: x[0])
    
    # Initialize the list for safe runs (valid periods)
    safe_runs = []

    # Iterate through the reward intervals
    for i in range(1, len(all_rewards)):
        start_ts, start_idx = all_rewards[i-1]
        end_ts, end_idx = all_rewards[i]
        
        # Check if this interval includes any air puff events
        exclude = False
        for puff_ts in airpuff_ts:
            if start_ts <= puff_ts <= end_ts:
                exclude = True
                break
        
        # If no air puff event is in the range, it's a valid interval (safe run)
        if not exclude:
            safe_runs.append(((start_ts, start_idx), (end_ts, end_idx)))
    
    return safe_runs

In [7]:
def plot_safe_runs(x_run, time_run, safe_runs, left_rewards, right_rewards):
    """
    Plots the x-location vs time and marks the valid reward locations for safe runs as lines.

    Parameters:
    - x_run: x location values (positions) for the run (as a pandas Series or numpy array).
    - time_run: Time values corresponding to the x location.
    - safe_runs: List of valid reward intervals (start, end).
    - left_rewards: List of left reward timestamps with indices.
    - right_rewards: List of right reward timestamps with indices.
    """
    # Plot x-location vs time for the whole run
    plt.figure(figsize=(10, 6))
    # plt.plot(time_run, x_run, label="X-Location over Time", color="blue", alpha=0.7)
    
    # Extract left and right reward locations
    left_reward_x_values = get_reward_x_values(x_run, left_rewards)
    right_reward_x_values = get_reward_x_values(x_run, right_rewards)

    left_reward_locations = [reward[1] for reward in left_reward_x_values]
    right_reward_locations = [reward[1] for reward in right_reward_x_values]
    
    left_reward_time = [ts for ts, idx in left_rewards]
    right_reward_time = [ts for ts, idx in right_rewards]

    # Plot left and right reward locations
    plt.scatter(left_reward_time, left_reward_locations, color='green', label="Left Reward", zorder=5)
    plt.scatter(right_reward_time, right_reward_locations, color='red', label="Right Reward", zorder=5)
    plt.scatter(puff_ts, puff_x_location, color='orange', s=50, label="Airpuff Events")

    
    # Loop through the safe runs and plot them as lines
    for (start_ts, start_idx), (end_ts, end_idx) in safe_runs:
        start_x = x_run.iloc[start_idx] if isinstance(x_run, pd.Series) else x_run[start_idx]
        end_x = x_run.iloc[end_idx] if isinstance(x_run, pd.Series) else x_run[end_idx]
        start_time = time_run[start_idx] if isinstance(time_run, np.ndarray) else time_run.iloc[start_idx]
        end_time = time_run[end_idx] if isinstance(time_run, np.ndarray) else time_run.iloc[end_idx]
        # Plot the line for this safe run
        plt.plot(time_run[start_idx: end_idx], x_run.iloc[start_idx:end_idx], color='blue', alpha=0.6, lw=2)
    
    # Customize the plot
    plt.title('X-Location Over Time with Safe Reward Intervals')
    plt.xlabel('Time (ms)')
    plt.ylabel('X-Location')
    plt.legend(loc='upper right')
    plt.show()

In [8]:
def exclude_airpuff_periods_matrix(left_rewards, right_rewards, airpuff_ts):
    """
    Exclude the periods where air puffs occurred between reward timestamps.
    
    Parameters:
    - left_rewards: List of left reward timestamps and indices.
    - right_rewards: List of right reward timestamps and indices.
    - airpuff_ts: List of air puff timestamps.
    
    Returns:
    - safe_runs: A numpy array where each row represents a safe run with columns
      [start_time, start_index, end_time, end_index] for each safe run.
    """
    # Combine left and right rewards and sort by timestamp
    all_rewards = sorted(left_rewards + right_rewards, key=lambda x: x[0])
    
    # Initialize the list for safe runs (valid periods)
    safe_runs = []

    # Iterate through the reward intervals
    for i in range(1, len(all_rewards)):
        start_ts, start_idx = all_rewards[i-1]
        end_ts, end_idx = all_rewards[i]
        
        # Check if this interval includes any air puff events
        exclude = False
        for puff_ts in airpuff_ts:
            if start_ts <= puff_ts <= end_ts:
                exclude = True
                break
        
        # If no air puff event is in the range, it's a valid interval (safe run)
        if not exclude:
            # Append the safe run as a row in the matrix [start_time, start_index, end_time, end_index]
            safe_runs.append([float(start_ts), start_idx, float(end_ts), end_idx])
    
    # Convert the safe_runs list to a numpy array (matrix)
    return np.array(safe_runs, dtype = object)

In [9]:
def get_x_values_from_idx(x_run, reward_ts):
    """
    Retrieve the x values corresponding to reward timestamps using their indices.

    Parameters:
    - x_run: A Pandas Series or NumPy array of x values (positions) for the run.
    - reward_ts_with_indices: List of tuples [(timestamp, index)] from preprocess_rewards.

    Returns:
    - reward_x_values: List of (timestamp, x_value) tuples.
    """
    reward_x_values = []

    for idx in reward_ts:
        # Ensure index is valid
        if 0 <= idx < len(x_run):
            x_value = x_run.iloc[idx]  # Use .iloc for positional indexing
            reward_x_values.append(x_value)
        else:
            reward_x_values.append(None)  # Handle invalid indices gracefully

    return reward_x_values

In [10]:
def calculate_middle_x(left_reward_x, right_reward_x):
    """
    Calculate the middle x position between each left and right reward location.

    Parameters:
    - left_reward_x: List of tuples (timestamp, x_value) for left rewards.
    - right_reward_x: List of tuples (timestamp, x_value) for right rewards.

    Returns:
    - middle_x_values: List of middle x positions between corresponding left and right rewards.
    """
    # Check that the lists are of equal length
    if len(left_reward_x) != len(right_reward_x):
        raise ValueError("Left and right reward lists must be of equal length")

    middle_x_values = []

    for (left_x), (right_x) in zip(left_reward_x, right_reward_x):
        # Ensure valid x values (not None)
        if left_x is not None and right_x is not None:
            middle_x = (left_x + right_x) / 2  # Calculate the halfway point
            middle_x_values.append(middle_x)
        else:
            middle_x_values.append(None)  # Handle invalid cases gracefully

    return middle_x_values

In [19]:
def calculate_one_third_x(left_reward_x, right_reward_x):
    """
    Calculate the middle x position between each left and right reward location.

    Parameters:
    - left_reward_x: List of tuples (timestamp, x_value) for left rewards.
    - right_reward_x: List of tuples (timestamp, x_value) for right rewards.

    Returns:
    - middle_x_values: List of middle x positions between corresponding left and right rewards.
    """
    # Check that the lists are of equal length
    if len(left_reward_x) != len(right_reward_x):
        raise ValueError("Left and right reward lists must be of equal length")

    middle_x_values = []

    for (left_x), (right_x) in zip(left_reward_x, right_reward_x):
        # Ensure valid x values (not None)
        if left_x is not None and right_x is not None:
            middle_x = (left_x + right_x) / 3  # Calculate the first third of the track
            middle_x_values.append(middle_x)
        else:
            middle_x_values.append(None)  # Handle invalid cases gracefully

    return middle_x_values

In [12]:
def find_closest_middle_x_indices(x_run, middle_x_values, safe_runs_matrix):
    """
    Find the indices of the closest x values to the middle x values within specified ranges.

    Parameters:
    - x_run: Pandas Series containing the x position data.
    - middle_x_values: List of middle x values between left and right rewards.
    - left_reward_x: List of tuples (timestamp, index) for left reward x values.
    - right_reward_x: List of tuples (timestamp, index) for right reward x values.

    Returns:
    - closest_indices: List of indices in x_run where x is closest to middle_x.
    """
    closest_indices = []

    for i, middle_x in enumerate(middle_x_values):
        # Get the start and end indices for the search range
        left_idx = safe_runs_matrix[i][1]  # Index of the left reward
        right_idx = safe_runs_matrix[i][3]   # Index of the right reward
        
        # Ensure we are slicing in the correct order
        if right_idx > left_idx:
            # If right_idx > left_idx, switch the range to avoid an empty sequence
            x_range = x_run.iloc[left_idx:right_idx + 1]  # Pandas subseries in the correct range
        else:
            x_range = x_run.iloc[right_idx:left_idx + 1]  # Pandas subseries in the correct range

        # Check if x_range is empty
        if x_range.empty:
            print(f"Warning: Empty range for indices {right_idx} to {left_idx}. Skipping.")
            continue
        
        # Find the closest value to middle_x in this range
        abs_diff = np.abs(x_range.to_numpy() - middle_x)  # Absolute difference
        closest_local_idx = abs_diff.argmin()  # Local index within the range

        # Convert local index to the global index in x_run
        global_idx = x_range.index[closest_local_idx]
        closest_indices.append(global_idx)

    return closest_indices

In [13]:
def find_closest_valid_indices(time_run, desired_indices):
    """
    Find the closest valid indices in the `time_run` Series/DataFrame for a list of `desired_indices`.
    If a desired index is not present, try the next closest index until a valid one is found.
    
    Parameters:
    - time_run: A pandas Series or DataFrame with a valid index.
    - desired_indices: A list of index values to look for in `time_run`.
    
    Returns:
    - closest_indices: A list of closest valid indices found in `time_run` for each desired index.
    """
    # Ensure the input is sorted (important for sequential checking)
    time_run = time_run.sort_index()
    
    closest_indices = []
    print(desired_indices)
    for desired_index in desired_indices:
        if desired_index in time_run.index:
            closest_indices.append(desired_index)  # If valid, add directly
        else:
            to_be_changed = desired_index
            while to_be_changed not in time_run.index:
                to_be_changed +=1
                if to_be_changed in time_run.index:
                    closest_indices.append(to_be_changed)
        # elif:
        #     # If no valid index exists above, raise an error
        #     raise ValueError(f"No valid index found in `time_run` above the desired index: {desired_index}")
    
    return closest_indices

In [15]:
def safe_runs_with_middle(safe_runs, x_run):
    left_reward_x = get_x_values_from_idx(x_run, safe_runs[:, 1])
    right_reward_x = get_x_values_from_idx(x_run, safe_runs[:, 3])
    middle_x = calculate_middle_x(left_reward_x, right_reward_x)
    closest_middle_idx = find_closest_middle_x_indices(x_run, middle_x, safe_runs)
    middle_time = time_run.loc[closest_middle_idx].tolist()

    final_safe_runs = []
    for i, val in enumerate (safe_runs):
        start_time, start_idx = safe_runs[i][0], safe_runs[i][1]
        end_time, end_idx = safe_runs[i][2], safe_runs[i][3]
        final_safe_runs.append([start_time, start_idx, middle_time[i], middle_x[i], end_time, end_idx])

    final_safe_runs = np.array(final_safe_runs, dtype = object)
    return final_safe_runs

In [20]:
def safe_runs_with_third(safe_runs, x_run):
    left_reward_x = get_x_values_from_idx(x_run, safe_runs[:, 1])
    right_reward_x = get_x_values_from_idx(x_run, safe_runs[:, 3])
    middle_x = calculate_one_third_x(left_reward_x, right_reward_x)
    closest_middle_idx = find_closest_middle_x_indices(x_run, middle_x, safe_runs)
    middle_time = time_run.loc[closest_middle_idx].tolist()

    final_safe_runs = []
    for i, val in enumerate (safe_runs):
        start_time, start_idx = safe_runs[i][0], safe_runs[i][1]
        end_time, end_idx = safe_runs[i][2], safe_runs[i][3]
        final_safe_runs.append([start_time, start_idx, middle_time[i], middle_x[i], end_time, end_idx])

    final_safe_runs = np.array(final_safe_runs, dtype = object)
    return final_safe_runs

In [16]:
def plot_safe_runs_with_middle(safe_runs_with_middle, x_run, time_run):
    left_reward_x = get_x_values_from_idx(x_run, safe_runs_with_middle[:, 1])
    right_reward_x = get_x_values_from_idx(x_run, safe_runs_with_middle[:, 5])
    middle_x = calculate_middle_x(left_reward_x, right_reward_x)
    #plot the left, right and middle points
    plt.scatter(safe_runs_with_middle[:, 0], left_reward_x, color='green', label="Left Reward", zorder=5)
    plt.scatter(safe_runs_with_middle[:, 4], right_reward_x, color='red', label="Right Reward", zorder=5)
    plt.scatter(safe_runs_with_middle[:, 2], middle_x, color='orange', label="Middle of track", zorder=5)
    #plot airpuffs as a check
    # plt.scatter(puff_ts, puff_x_location, color='yellow', s=50, label="Airpuff Events")

    for i,val in enumerate(safe_runs_with_middle):
        start_idx = safe_runs_with_middle[i][1]
        end_idx = safe_runs_with_middle[i][5]
        plt.plot(time_run[start_idx: end_idx], x_run.iloc[start_idx:end_idx], color='blue', alpha=0.6, lw=2)

    plt.title('X-Location Over Time with Safe Reward Intervals and Middle of the Track')
    plt.xlabel('Time (ms)')
    plt.ylabel('X-Location')
    plt.legend(bbox_to_anchor =(0.8, -0.13), ncol = 2)
    plt.show()

In [1]:
def plot_safe_runs_with_third(safe_runs_with_middle, x_run, time_run):
    left_reward_x = get_x_values_from_idx(x_run, safe_runs_with_middle[:, 1])
    right_reward_x = get_x_values_from_idx(x_run, safe_runs_with_middle[:, 5])
    middle_x = calculate_one_third_x(left_reward_x, right_reward_x)
    #plot the left, right and middle points
    plt.scatter(safe_runs_with_middle[:, 0], left_reward_x, color='green', label="Left Reward", zorder=5)
    plt.scatter(safe_runs_with_middle[:, 4], right_reward_x, color='red', label="Right Reward", zorder=5)
    plt.scatter(safe_runs_with_middle[:, 2], middle_x, color='orange', label="Middle of track", zorder=5)
    #plot airpuffs as a check
    # plt.scatter(puff_ts, puff_x_location, color='yellow', s=50, label="Airpuff Events")

    for i,val in enumerate(safe_runs_with_middle):
        start_idx = safe_runs_with_middle[i][1]
        end_idx = safe_runs_with_middle[i][5]
        plt.plot(time_run[start_idx: end_idx], x_run.iloc[start_idx:end_idx], color='blue', alpha=0.6, lw=2)

    plt.title('X-Location Over Time with Safe Reward Intervals and Middle of the Track')
    plt.xlabel('Time (ms)')
    plt.ylabel('X-Location')
    plt.legend(bbox_to_anchor =(0.8, -0.13), ncol = 2)
    plt.show()